# Stage 1: Chunking Strategies Comparison

This notebook compares the 4 chunking strategies on the PubMed corpus:
- **Fixed** (baseline): 200-token chunks with 50-token overlap
- **Semantic**: Split at cosine similarity drops between adjacent sentences (PubMedBERT)
- **Section-aware**: Parse PubMed abstract section headers (BACKGROUND/METHODS/RESULTS/CONCLUSIONS)
- **Recursive**: LangChain RecursiveCharacterTextSplitter with metadata enrichment

We measure: chunk count, average chunk size, size distribution, and a qualitative look at chunk boundaries.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1.1 Load the corpus

In [ ]:
from src.shared.corpus_loader import load_corpus

corpus = load_corpus("data/corpus.json")
print(f"Corpus: {len(corpus)} articles")
print(f"Avg abstract length: {sum(len(a['abstract'].split()) for a in corpus) / len(corpus):.0f} words")
print(f"\nSample article:")
print(f"  PMID: {corpus[0]['pmid']}")
print(f"  Title: {corpus[0]['title'][:80]}...")
print(f"  Abstract: {corpus[0]['abstract'][:200]}...")

## 1.2 Run all 4 chunking strategies

In [ ]:
from src.chunking import chunk_corpus
import time

strategies = ["fixed", "semantic", "section_aware", "recursive"]
results = {}

for strategy in strategies:
    start = time.time()
    chunks = chunk_corpus(corpus, strategy=strategy)
    elapsed = time.time() - start
    
    word_counts = [len(c["text"].split()) for c in chunks]
    results[strategy] = {
        "chunks": chunks,
        "count": len(chunks),
        "avg_words": sum(word_counts) / len(word_counts),
        "min_words": min(word_counts),
        "max_words": max(word_counts),
        "time_seconds": round(elapsed, 2),
    }
    print(f"{strategy:>15}: {len(chunks):>4} chunks | avg {results[strategy]['avg_words']:.0f} words | "
          f"range [{results[strategy]['min_words']}-{results[strategy]['max_words']}] | {elapsed:.2f}s")

## 1.3 Summary table

In [ ]:
summary = pd.DataFrame([
    {
        "Strategy": s,
        "Chunks": results[s]["count"],
        "Avg Words": round(results[s]["avg_words"]),
        "Min Words": results[s]["min_words"],
        "Max Words": results[s]["max_words"],
        "Time (s)": results[s]["time_seconds"],
    }
    for s in strategies
])
summary

## 1.4 Chunk size distribution

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for ax, strategy, color in zip(axes, strategies, colors):
    word_counts = [len(c["text"].split()) for c in results[strategy]["chunks"]]
    ax.hist(word_counts, bins=20, color=color, edgecolor="white", alpha=0.8)
    ax.set_title(strategy.replace("_", " ").title())
    ax.set_xlabel("Words per chunk")
    ax.axvline(sum(word_counts)/len(word_counts), color="red", linestyle="--", alpha=0.7, label="mean")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Frequency")
plt.suptitle("Chunk Size Distribution by Strategy", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 1.5 Chunks per article distribution

In [ ]:
from collections import Counter

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.2

for i, (strategy, color) in enumerate(zip(strategies, colors)):
    chunks = results[strategy]["chunks"]
    pmid_counts = Counter(c["pmid"] for c in chunks)
    counts = list(pmid_counts.values())
    vals, freqs = zip(*sorted(Counter(counts).items()))
    ax.bar([v + i * width for v in vals], freqs, width=width, label=strategy, color=color, alpha=0.8)

ax.set_xlabel("Chunks per article")
ax.set_ylabel("Number of articles")
ax.set_title("How many chunks does each article produce?")
ax.legend()
plt.tight_layout()
plt.show()

## 1.6 Qualitative comparison — same article, different strategies

In [ ]:
# Pick the first article and show how each strategy chunks it
sample_pmid = corpus[0]["pmid"]
print(f"Article PMID: {sample_pmid}")
print(f"Title: {corpus[0]['title']}")
print(f"Abstract length: {len(corpus[0]['abstract'].split())} words")
print("=" * 80)

for strategy in strategies:
    article_chunks = [c for c in results[strategy]["chunks"] if c["pmid"] == sample_pmid]
    print(f"\n--- {strategy.upper()} ({len(article_chunks)} chunks) ---")
    for c in article_chunks:
        meta = c.get("metadata", {})
        meta_str = f" [{meta}]" if meta else ""
        print(f"  Chunk {c['chunk_index']}{meta_str}: {c['text'][:120]}...")
        print(f"    ({len(c['text'].split())} words)")

## 1.7 Section-aware: section distribution

In [ ]:
# How many chunks fall into each section type?
section_chunks = results["section_aware"]["chunks"]
section_counts = Counter(c.get("metadata", {}).get("section", "unknown") for c in section_chunks)

print("Section distribution (section-aware chunking):")
for section, count in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"  {section:>20}: {count:>3} chunks ({count/len(section_chunks)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
sections = sorted(section_counts.keys())
ax.barh(sections, [section_counts[s] for s in sections], color="#55A868")
ax.set_xlabel("Number of chunks")
ax.set_title("Section-Aware: Chunks by Section Type")
plt.tight_layout()
plt.show()

## 1.8 Recursive: metadata enrichment

In [ ]:
# What study types were detected?
recursive_chunks = results["recursive"]["chunks"]
study_types = Counter(c.get("metadata", {}).get("study_type", "unknown") for c in recursive_chunks)

print("Study type distribution (recursive chunking with metadata):")
for st, count in sorted(study_types.items(), key=lambda x: -x[1]):
    print(f"  {st:>20}: {count:>3} chunks")

# How many chunks have sample sizes?
with_sample_size = sum(1 for c in recursive_chunks if c.get("metadata", {}).get("sample_size"))
print(f"\nChunks with sample size extracted: {with_sample_size}/{len(recursive_chunks)}")

## Key Takeaways

| Strategy | Chunks | Avg Size | Key Property |
|----------|--------|----------|-------------|
| Fixed | ~84 | ~150 words | Uniform size, may split mid-sentence |
| Semantic | ~94 | ~130 words | Respects topic boundaries, variable size |
| Section-aware | ~84 | ~150 words | Preserves methodological context |
| Recursive | ~144 | ~85 words | Smaller chunks with rich metadata |

**Next step:** Stage 2 tests how these different chunk sets affect retrieval quality.